In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='17828a10-3c5c-4f94-8f8f-e8a5fcca9e11'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "what is langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'call_357363': 'EnMKcQFpFH0Tx3nIKDDsDuA9QSr6cXwuwOkYFhKtzhwsAxn/CVF7tkZR4uOFyeFszss176yaIeUbMwd0lpinDJUQ+Bk3WJXPTVemAlKVlfIHAGanhAGSK6JTHntms53HVktMZfFEpYnY88ulunQ3X84dmaBv'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fe10-72c1-7583-9816-e6cc36505cc0-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'what is langchain-mcp-adapters library'}, 'id': 'call_357363', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 24, 'total_tokens': 223, '

## Online MCP

In [8]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [9]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
)

In [10]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='42ace77c-03f5-4f6c-aa6d-1d200de57e30'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "America/New_York"}'}, '__gemini_function_call_thought_signatures__': {'call_311304': 'EnMKcQFpFH0Tm2QBVCL5HwFxk1cPk3v6EdXinR5DhR7vu1hYlaaPp3JxLyn35lxKHt9ezGsNEnekEqsoTaCXkrMJK4AM4s+kwt0KyuQlgK9fyEkm9fNizF2QOaIAK47iM6BSRkgo9jWoL15GTfcmpGkJijwp'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fe11-3ba5-7251-b2fe-47e5ed984fce-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'call_311304', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 22, 'total_tokens': 306, 'input_token_details': {'cache_read': 0}}),
              T